# RQ3: training data and exposure — training

E7 sampling, E8 corpus eligibility and E9 episode size address distinct changes in data exposure. E8/E9 are PD-only. The exact large-table 8,192-row control is reused once. Equal optimizer updates do not imply equal compute or unique rows.

## 0. Evidence and interpretation

The selected local evidence is a snapshot. Missing evidence remains unavailable; queued VSC jobs are not completed results.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, questions as rq, synthesis as sy, research_views as rv
from src.visualize import choice_effects as ce, comparisons as cmp, publication as pub, dashboards as db
from src.visualize import diagnostics as dg
from src.visualize import training_summary as ts, pilots as pv, finishing as finish, prediction_diagnostics as pred
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
sink = FigureSaver('rq3/01_training')
report = cp.NotebookReport('RQ3: training data and exposure — training', sink=sink)
print(source_description())
report.add('0. Evidence and interpretation', source_description() + '\n' + 'E7 sampling, E8 corpus eligibility and E9 episode size address distinct changes in data exposure. E8/E9 are PD-only. The exact large-table 8,192-row control is reused once. Equal optimizer updates do not imply equal compute or unique rows.')

## 1. Design and complete training coverage

Historical artifact identities remain unchanged. Additional v2/v2.6 runs join these views only when their evidence is in the selected input. Shared trial identities are counted once.

In [ ]:
QUESTION = 3
studies = {track: rq.load(QUESTION, track) for track in ('pd', 'lgd')}
training = {track: rq.training_studies(values, QUESTION) for track, values in studies.items()}
report.table('Included scientific analyses', rv.design_table().query('rq == @QUESTION'))
for track, values in training.items():
    report.table(track.upper() + ' unique training trials', rq.coverage(values))
    report.table(track.upper() + ' recipe differences and actual inference policy', sy.comparability_table(values))
report.add('1. Design and complete training coverage', 'Historical artifact identities remain unchanged. Additional v2/v2.6 runs join these views only when their evidence is in the selected input. Shared trial identities are counted once.')

## 2. Sampling, corpus and episode trajectories

E7 sampling and the E8/E9 corpus and episode-size arms are shown in separate figures to preserve their interpretation. The large-table 8,192-row curve is the shared E8/E9 control. Monitor rows describe fixed-context monitoring on excluded tables.

In [ ]:
for track, values in studies.items():
    for title, name, campaigns in [('Training trajectories by sampling policy', 'sampling_trajectories', (3,)),
                                   ('Training trajectories by corpus and episode size', 'corpus_episode_trajectories', (6, 7))]:
        subset = [item for item in values if item[0] in campaigns]
        if subset:
            cp.show(sink, rq.plot_training(subset, title=title, name=name), track=track)
report.add('2. Sampling, corpus and episode trajectories', 'E7 sampling and the E8/E9 corpus and episode-size arms are shown in separate figures to preserve their interpretation. The large-table 8,192-row curve is the shared E8/E9 control. Monitor rows describe fixed-context monitoring on excluded tables.')

## 3. Exposure is different from update count

The same sampling curves are indexed by processed row exposures, including repeated rows. This reveals the additional data and computation behind a fixed update budget; it is not a unique-row count.

In [ ]:
for track, values in studies.items():
    sample = [item for item in values if item[0] == 3]
    cp.show(sink, rq.plot_training(sample, ('credit', 'retention'), title='Monitor changes per processed row exposure', name='sampling_exposure', row_exposure=True), track=track)
    for e, arm, run in sample:
        report.table(track.upper() + ' sampling cost and exposure', cmp.sampling_costs(run))
report.add('3. Exposure is different from update count', 'The same sampling curves are indexed by processed row exposures, including repeated rows. This reveals the additional data and computation behind a fixed update budget; it is not a unique-row count.')

## 4. Optimization health, parameter movement and measured cost

Record the complete optimizer diagnostics and resources without duplicating nearly identical plots. Check missing telemetry, update counts and numerical skips before interpreting trajectories. Parameter movement is not predictive bias; memory is a measured resource, not a scientific outcome.

In [ ]:
for track, values in training.items():
    report.table(track.upper() + ' cost by recipe', sy.costs(values))
    for e, arm, run in values:
        label = track.upper() + ' ' + str(run.cfg.run_name) + ' ' + str(arm or '')
        report.table(label + ' optimization diagnostics', cp.diagnostics(run))
        report.table(label + ' completed training cost', pub.cost_summary(run))
        report.table(label + ' parameter movement', ts.movement_tables(run))
report.add('4. Optimization health, parameter movement and measured cost', 'Record the complete optimizer diagnostics and resources without duplicating nearly identical plots. Check missing telemetry, update counts and numerical skips before interpreting trajectories. Parameter movement is not predictive bias; memory is a measured resource, not a scientific outcome.')

## Complete text summary

Every displayed table, plotted value and paper caption, in section order.

In [ ]:
print(report.summary(sink))